# Data modeling 4: relationships and double counting

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-04-relationships-and-double-counting.ipynb)

Most wrong numbers in analytics are not bad arithmetic. They come from a join that matched more rows than
expected, so a measure was counted two, three or eight times. This lesson names the patterns that cause it
(fan-out, the chasm trap, the fan trap), shows how to detect them, and models the relationships that cause them
properly: **bridge tables** for many-to-many relationships and hierarchy tables for trees.

**Prerequisites:** [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb).

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_products` (the product catalogue), `src_orders` (one row per order, with fulfilment timestamps), `src_order_lines` (one row per product on an order), `src_payments` (one row per payment; an order can have several) and `src_promotions` and `src_order_promotions` (promotions applied to orders): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
create table src_payments (payment_id varchar primary key, order_id int, paid_ts timestamp, method varchar, amount decimal(10, 2));
insert into src_payments values  -- order 1003 was paid in two parts; 1008 is unpaid and 1009 was cancelled
    ('P01', 1001, '2026-01-05 09:13', 'card',    76.99),
    ('P02', 1002, '2026-01-09 14:31', 'card',    60.00),
    ('P03', 1003, '2026-01-14 11:06', 'card',   132.50),
    ('P04', 1003, '2026-02-13 00:00', 'invoice', 100.00),
    ('P05', 1004, '2026-01-28 16:41', 'card',    34.99),
    ('P06', 1005, '2026-02-03 10:21', 'card',   114.99),
    ('P07', 1006, '2026-03-13 00:00', 'invoice', 400.00),
    ('P08', 1007, '2026-02-14 09:51', 'cash',    54.00),
    ('P09', 1010, '2026-02-27 08:31', 'cash',    33.00);
create table src_promotions (promo_code varchar primary key, promo_name varchar);
insert into src_promotions values ('WINTER10', 'Winter sale'), ('BULK', 'Bulk discount'), ('LOYAL', 'Loyalty reward');
create table src_order_promotions (order_id int, promo_code varchar, primary key (order_id, promo_code));
insert into src_order_promotions values (1002, 'WINTER10'), (1003, 'BULK'), (1003, 'LOYAL'), (1006, 'BULK');
""")
con.sql("show tables").show()

The next cell builds the order-line fact used throughout the lesson.

In [ ]:
con.sql("""
create table fact_sales as
select l.order_id, l.line_no, o.customer_id, l.sku, cast(o.order_ts as date) as order_date,
       l.qty as quantity, l.qty * l.unit_price - l.discount as net_amount
from src_order_lines l join src_orders o using (order_id)
where o.status <> 'cancelled'
""")
con.sql("select count(*) as fact_rows, sum(net_amount) as net_sales from fact_sales").show()

## Cardinality and fan-out

Every join has a **cardinality**: how many rows on each side match one row on the other. Fact-to-dimension
joins must be many-to-one. If the "one" side has two rows for a key, each matching fact row comes out twice.
This is **fan-out**, and it inflates every additive measure without any error.

Fan-out sneaks in when a table that looks like a dimension is not unique on the join key: a customer table with
one row per address, a Type 2 dimension joined without its date condition, a mapping table someone appended to
twice. Referential integrity (every fact key exists in the dimension) does not imply uniqueness.

Two habits catch it:

- Before joining, check that the key is unique on the side you expect to be "one":
  `select key from t group by key having count(*) > 1` should return nothing.
- After joining, compare the row count and a total with the fact alone.

**Where it matters**
- Every new join in a model, and every model test suite (a uniqueness test on each dimension key).

**Watch out for**
- `select distinct` added to hide duplicates: it hides the symptom and can also merge rows that were legitimately separate.

### Example

Customers can have more than one account manager. Customer 101 is shared by two managers. Join sales to the
manager table to report sales by manager region, and check the totals.

In [ ]:
con.sql("""
create table customer_managers (customer_id int, manager varchar, region varchar);
insert into customer_managers values
    (101, 'Priya', 'North'), (101, 'Tom', 'South'),   -- a shared account
    (102, 'Priya', 'North'), (103, 'Ana', 'West'), (104, 'Tom', 'South'), (105, 'Ana', 'West');
""")
con.sql("select customer_id, count(*) as managers from customer_managers group by all having count(*) > 1").show()
con.sql("""
select (select sum(net_amount) from fact_sales) as true_total,
       sum(f.net_amount)                        as total_after_join,
       count(*)                                 as rows_after_join
from fact_sales f join customer_managers m using (customer_id)
""").show()

**What to notice:** The uniqueness check finds customer 101 with two managers. Joining through that table turns 14 fact rows into 19
and 1,017.50 into 1,152.50: each of customer 101's 5 lines (135.00) is counted twice. The data is right; the
relationship is many-to-many, and it needs a bridge with weights (later in this lesson) or a rule that picks one
primary manager.

## The chasm trap: two facts joined through a shared key

A **chasm trap** happens when two one-to-many relationships hang off the same parent and you join both at once:
an order has many lines and many payments. Each line repeats for every payment of its order and each payment
repeats for every line. Both sums are wrong, and differently wrong for each order.

The fix is the drill-across from lesson 2: **aggregate each child to the parent's grain first**, then join
the aggregates, which are one row per order on both sides.

**Where it matters**
- Orders with lines and payments, invoices with lines and credit notes, tickets with comments and status changes.

**Watch out for**
- Totals that look right for most rows: the trap only inflates parents with more than one child on both sides, here only order 1003.

### Example

Sales and payments per order, joined naively and then aggregated first.

In [ ]:
con.sql("""
select f.order_id, sum(f.net_amount) as net_sales, sum(p.amount) as paid
from fact_sales f join src_payments p using (order_id)
where f.order_id in (1001, 1003)
group by f.order_id order by f.order_id
""").show()

In [ ]:
con.sql("""
with sales as (select order_id, sum(net_amount) as net_sales from fact_sales group by order_id),
     paid  as (select order_id, sum(amount)     as paid      from src_payments group by order_id)
select order_id, s.net_sales, p.paid
from sales s left join paid p using (order_id)
where order_id in (1001, 1003)
order by order_id
""").show()

**What to notice:** In the naive join, order 1001 looks plausible but its payment of 76.99 has doubled to 153.98, because the order
has two lines. Order 1003 has two lines and two payments, so both measures doubled: sales 465.00 instead of 232.50
and payments 465.00 instead of 232.50. Aggregating each side to one row per order first gives the true values.

## The fan trap: filtering a parent by its children

A **fan trap** is the one-sided version: you join a parent-grain table to a child table only to *filter*
("orders that contain coffee"), then sum a parent-level measure. An order with two coffee lines is counted
twice.

When the child table is only a filter, use `exists` (a **semi-join**). It keeps each parent row once if at
least one child matches, however many do.

**Where it matters**
- "Customers who bought X", "orders containing a discounted line", "accounts with any overdue invoice".

**Watch out for**
- Fixing it with `count(distinct order_id)`: that repairs a count but not a sum of shipping fees or order totals.

### Example

Total shipping fees on orders that contain a Grocery product, with a join and with `exists`.

In [ ]:
con.sql("""
select 'join' as method, count(*) as orders_counted, sum(o.shipping_fee) as shipping_fees
from src_orders o
join src_order_lines l using (order_id)
join src_products p using (sku)
where p.department = 'Grocery' and o.status <> 'cancelled'
union all
select 'exists', count(*), sum(o.shipping_fee)
from src_orders o
where o.status <> 'cancelled'
  and exists (select 1 from src_order_lines l join src_products p using (sku)
              where l.order_id = o.order_id and p.department = 'Grocery')
""").show()

**What to notice:** Orders 1003 and 1005 each have a coffee line and a tea line, so the join counts both twice: 8
"orders" instead of 6. Order 1003 ships free, so only 1005's fee is doubled, and fees come to 19.96 instead of
14.97. `exists` keeps each qualifying order once, however many of its lines match.

## Bridge tables for many-to-many relationships

Some relationships are genuinely many-to-many: an order can use several promotions and a promotion applies to
many orders; an account has several owners; a patient has several diagnoses. A **bridge table** resolves it
with one row per pair (`order_id`, `promo_code`) between the fact and the dimension.

Two different questions come out of a bridge, and they must not be confused:

- **Full credit** (impact): "how much revenue came from orders that used BULK?" Each order counts in full for
  every promotion it used. The per-promotion figures are each correct, but they add up to more than total
  revenue.
- **Allocated** (attribution): "how much revenue should we attribute to each promotion?" Give each bridge row a
  **weighting factor**, so that each order's weights sum to 1, and multiply. Allocated figures add up to the
  true total.

Keep the weight on the bridge so the allocation rule is decided once. Test that each parent's weights sum to
exactly 1.

**Where it matters**
- Promotions, marketing touches, multiple diagnoses, joint accounts, products in several categories.

**Watch out for**
- Presenting full-credit figures next to a grand total: readers add them up and find more revenue than exists.
- Weights that do not sum to 1 per parent after someone adds a bridge row without recomputing them.

### Example

Build a weighted bridge between orders and promotions, then report revenue by promotion both ways.

In [ ]:
con.sql("""
create table bridge_order_promotion as
select order_id, promo_code, 1.0 / count(*) over (partition by order_id) as weight  -- an even split
from src_order_promotions;
""")
con.sql("select * from bridge_order_promotion order by order_id, promo_code").show()
con.sql("""
with orders as (select order_id, sum(net_amount) as order_revenue from fact_sales group by order_id)
select coalesce(b.promo_code, '(all promotions)') as promo_code,
       sum(o.order_revenue)             as full_credit_revenue,
       cast(sum(o.order_revenue * b.weight) as decimal(12, 2)) as allocated_revenue
from bridge_order_promotion b join orders o using (order_id)
group by rollup (b.promo_code)
order by grouping(b.promo_code), b.promo_code
""").show()

In [ ]:
con.sql("""
select (select sum(net_amount) from fact_sales
        where order_id in (select order_id from src_order_promotions)) as revenue_of_promoted_orders
""").show()

**What to notice:** Order 1003 used BULK and LOYAL, so each gets a weight of 0.5. With full credit both promotions are credited with
the whole 232.50; the column totals 925.00 although the promoted orders only brought in 692.50. The allocated
column splits order 1003 in half and adds up to exactly 692.50. Both columns are useful; a report must say which
one it shows.

## Hierarchies: fixed and ragged

A **fixed-depth hierarchy** has the same levels everywhere (department → category → product). Store each level
as a column on the dimension (the star way) and roll up with `group by rollup`, which adds subtotal rows for
each level and a grand total. `grouping()` tells a subtotal row from a group whose value is really null.

A **ragged** or **parent-child** hierarchy has branches of different depths: an org chart, a chart of accounts,
a category tree where some branches have an extra level. Columns per level do not fit. Store the tree as
`(node, parent)` pairs and derive a **closure table** with one row per (ancestor, descendant) pair, including
each node with itself, using a recursive query. Joining facts through the closure table rolls them up to any
node in one ordinary `group by`.

**Where it matters**
- Product and category trees, cost centres, sales territories, organization charts, bills of materials.

**Watch out for**
- Rolling up through a parent-child table with one self-join per level: it breaks as soon as the tree gets deeper.
- A node with two parents: then the tree is a graph, and sums through the closure table double count unless weighted, like a bridge.

### Example

A fixed hierarchy: net sales by department and category with subtotals.

In [ ]:
con.sql("""
select case when grouping(p.department) = 1 then 'All departments' else p.department end as department,
       case when grouping(p.category) = 1   then '(subtotal)'      else p.category   end as category,
       sum(f.net_amount) as net_sales
from fact_sales f join src_products p using (sku)
group by rollup (p.department, p.category)
order by grouping(p.department), p.department, grouping(p.category), p.category
""").show()

**What to notice:** `rollup (department, category)` produces the category rows, a subtotal per department (Grocery 777.50, Kitchen
240.00) and one grand total of 1,017.50, all in a single pass. `grouping()` returns 1 on the rows where a column
was rolled up, which is how the query labels them.

### Example

A ragged hierarchy: the shop's merchandising tree puts Coffee and Tea under an extra "Hot drinks" level, while
Mugs and Kettles hang directly under Kitchen. Build the closure table and roll sales up to every node.

In [ ]:
con.sql("""
create table category_tree (node varchar, parent varchar);
insert into category_tree values
    ('All products', null), ('Grocery', 'All products'), ('Kitchen', 'All products'),
    ('Hot drinks', 'Grocery'), ('Coffee', 'Hot drinks'), ('Tea', 'Hot drinks'),
    ('Mugs', 'Kitchen'), ('Kettles', 'Kitchen');

create table category_closure as
with recursive walk(ancestor, descendant, depth) as (
    select node, node, 0 from category_tree                       -- every node is its own ancestor
    union all
    select w.ancestor, t.node, w.depth + 1
    from walk w join category_tree t on t.parent = w.descendant  -- step one level down
)
select * from walk;
""")
con.sql("select * from category_closure where ancestor in ('Grocery', 'Kitchen') order by ancestor, depth, descendant").show()

In [ ]:
con.sql("""
select c.ancestor as node, max(t2.depth) as level, sum(f.net_amount) as net_sales
from fact_sales f
join src_products p using (sku)
join category_closure c on c.descendant = p.category
join (select descendant as node, max(depth) as depth from category_closure
      where ancestor = 'All products' group by all) t2 on t2.node = c.ancestor
group by c.ancestor
order by level, node
""").show()

**What to notice:** The closure table lists Grocery's descendants at depth 1 (Hot drinks) and depth 2 (Coffee, Tea), but Kitchen's
leaves at depth 1: the branches have different depths and no fixed set of columns describes both. Joining the
fact through the closure table credits each sale to its category and every ancestor, so one `group by` gives
every node's total: Hot drinks 777.50 (all of Grocery's sales), Kitchen 240.00, All products 1,017.50.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A join of `fact_sales` to a customer table raised the row count from 14 to 19. What do you check first?

**A.** Whether the customer table is unique on the join key. Find the keys with `count(*) > 1`; those customers' facts were duplicated.

**Q2.** You need sales and refunds per order, each with several rows per order. Write the shape of the query.

**A.** Aggregate sales by order in one CTE, refunds by order in another, then join the two aggregates on `order_id` (a full join if either side can be missing).

**Q3.** How do you count the orders that contain at least one discounted line without double counting?

**A.** `select count(*) from orders o where exists (select 1 from order_lines l where l.order_id = o.order_id and l.discount > 0)`.

**Q4.** A promotion report shows each promotion's revenue, and the column adds up to more than total revenue. Is it wrong?

**A.** Not necessarily: it is full-credit revenue, where an order with two promotions counts for both. It is wrong only if presented as attribution. For attribution, use bridge weights that sum to 1 per order.

**Q5.** Why does an org chart need a closure table instead of `manager_level_1`, `manager_level_2` columns?

**A.** Reporting lines have different depths and change over time. A closure table handles any depth, and rolling up to any manager is one join and a `group by`.

## Key takeaways

- Fact-to-dimension joins must be many-to-one: check key uniqueness before the join and totals after it.
- Chasm trap: two child tables of one parent joined together multiply each other; aggregate each to the parent first.
- Fan trap: filter a parent by its children with `exists`, not a join.
- Model many-to-many relationships with a bridge; report full credit or weighted allocation, and say which.
- Fixed hierarchies are columns plus `rollup`; ragged hierarchies are a parent-child table plus a closure table.

**Next:** Practise lessons 1 to 4 in the [Data modeling drills](data-modeling-drills.ipynb), then continue with [Data modeling 5: normalized and operational models](data-modeling-05-normalized-and-operational-models.ipynb).